Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Maps with Altair

The same maps as the Week 7 Vega-Lite examples, made in Python. We build up in four steps:

1. **Base maps**: draw shapes from a map file (TopoJSON).
2. **Point maps**: put points on top of a base map, with **layers**.
3. **Choropleths**: colour each area by a value, with a **lookup**.
4. **Devolution**: England's strategic authorities, and zooming in on one.

The map files come from our [map-data repository](https://github.com/jhellingsdata/map-data); the data files are in the course repository (`weeks/week07/data`).

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import altair as alt

MAPS = "https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/"     # map files
DATA = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week07/data/"  # data files

<br><br>

---

<br>

## 1. Base maps

A **TopoJSON** file holds shapes. `alt.topo_feature()` loads one: give it the link, and the name of the group of shapes inside the file. In all our map-data files, the group is called `"geog"`.

`mark_geoshape()` draws the shapes, and `.project()` sets the **projection**: how the round Earth is flattened onto the page.

In [ ]:
wales = alt.topo_feature(MAPS + "gbr/CTRY_GB-WLS.json", "geog")

alt.Chart(wales).mark_geoshape(fill="#9bd3ae", stroke="white").project("mercator").properties(
    width=300, height=350, title="Wales"
)

<br>

More detail: the 22 local authorities of Wales. Each shape has **properties** (an `id` code and a `name`), which we can show in a tooltip. Hover over the map:

In [ ]:
wales_councils = alt.topo_feature(MAPS + "gbr/LAD_WLS_2026_05.json", "geog")

alt.Chart(wales_councils).mark_geoshape(fill="#9bd3ae", stroke="white").encode(
    tooltip=[alt.Tooltip("properties.name:N").title("Local authority"),
             alt.Tooltip("properties.id:N").title("Code")],
).project("mercator").properties(width=300, height=350, title="Wales: local authorities")

<br><br>

---

<br>

## 2. Point maps: layers

Point data is a table with a **latitude** and **longitude** for each row. On its own, the points float in space:

In [ ]:
airports = alt.Chart(DATA + "uk_airports.csv").mark_circle(color="#d35400", opacity=0.8).encode(
    longitude="longitude:Q",
    latitude="latitude:Q",
    size=alt.Size("passengers:Q").title("Passengers"),
    tooltip=[alt.Tooltip("airport:N").title("Airport"),
             alt.Tooltip("passengers:Q").format(",").title("Passengers")],
)

airports.project("mercator").properties(width=300, height=400)

<br>

So we draw a base map first, then the points on top. In Altair, `+` puts one chart on top of another (a **layer**). The projection goes on the layered chart, so both layers share it and line up:

In [ ]:
uk = alt.topo_feature(MAPS + "gbr/CTRY_UK.json", "geog")
base = alt.Chart(uk).mark_geoshape(fill="#e3e3e3", stroke="white")

(base + airports).project("mercator").properties(width=300, height=400, title="UK airports")

<br>

Two encodings at once: **colour** for fuel type and **size** for capacity. A square-root (`sqrt`) scale makes each circle's *area* match its value:

In [ ]:
plants = alt.Chart(DATA + "uk_power_plants.csv").mark_circle(opacity=0.7, stroke="white", strokeWidth=0.4).encode(
    longitude="longitude:Q",
    latitude="latitude:Q",
    color=alt.Color("fuel_category:N").title("Fuel"),
    size=alt.Size("capacity_mw:Q").title("Capacity (MW)").scale(type="sqrt", range=[5, 300]),
    tooltip=[alt.Tooltip("name:N").title("Power plant"),
             alt.Tooltip("fuel_category:N").title("Fuel"),
             alt.Tooltip("capacity_mw:Q").format(",.0f").title("Capacity (MW)")],
)

(base + plants).project("mercator").properties(width=350, height=450, title="UK power plants")

<br><br>

---

<br>

## 3. Choropleths: lookups

A **choropleth** colours each area by a value. The values are in a separate CSV file, so we need to join the two:
- the **map** has a code for each area: `properties.id` (e.g. `W06000015`, Cardiff);
- the **CSV** has the same codes in a column called `geo_id`, and a `value` for each.

`transform_lookup()` matches them, and copies the `value` from the CSV onto each shape. The CSV must have **one row per area**.

In [ ]:
grey_layer = alt.Chart(wales_councils).transform_calculate(
    no_data="'No data'"                                                            # text for the tooltip
).mark_geoshape(fill="#e3e3e3", stroke="white").encode(
    tooltip=[alt.Tooltip("properties.name:N").title("Local authority"),
             alt.Tooltip("no_data:N").title("Wages (£)")],
)

wages = alt.Chart(wales_councils).transform_lookup(
    lookup="properties.id",                                                        # the code in the map
    from_=alt.LookupData(DATA + "wales_wages_2024.csv", key="geo_id", fields=["value"]),  # the code in the CSV, and what to copy
).mark_geoshape(stroke="white").encode(
    color=alt.Color("value:Q").title("£ a year").scale(scheme="blues"),
    tooltip=[alt.Tooltip("properties.name:N").title("Local authority"),
             alt.Tooltip("value:Q").format(",.0f").title("Wages (£)")],
)

(grey_layer + wages).project("mercator").properties(width=320, height=380, title="Wages in Wales, 2024")

<br>

> **Why is one area grey?** There's no 2024 value for Blaenau Gwent in the data. Altair (Vega-Lite) leaves out shapes with a missing value, so on its own the coloured layer would have a hole there. The grey layer underneath (the same idea as the base map in the lecture) fills the gap, and its tooltip says "No data" instead of showing nothing. Showing missing data clearly is part of honest mapping.

### When the CSV has many years

`wages_LAD_WLS.csv` has a row for every area **and** year. A lookup keeps only **one** match per area (whichever comes last), so you'd get the wrong year, or nothing. Instead, flip it round:
1. start with the **CSV** and **filter** to one year;
2. look up each area's **shape** from the map, and store it as `geo`;
3. draw `geo` with the `shape` encoding.

In [ ]:
wages_by_year = alt.Chart(DATA + "wages_LAD_WLS.csv").transform_filter(
    "datum.date == 2022"
).transform_lookup(
    lookup="geo_id",
    from_=alt.LookupData(wales_councils, key="properties.id"),
    as_="geo",
).mark_geoshape(stroke="white").encode(
    shape=alt.Shape("geo:G"),          # G = geojson: draw the shape stored in 'geo'
    color=alt.Color("value:Q").title("£ a year").scale(scheme="blues"),
    tooltip=[alt.Tooltip("geo.properties.name:N").title("Local authority"),
             alt.Tooltip("value:Q").format(",.0f").title("Wages (£)")],
)

grey_councils = alt.Chart(wales_councils).mark_geoshape(fill="#e3e3e3", stroke="white")

(grey_councils + wages_by_year).project("mercator").properties(width=320, height=380, title="Wages in Wales, 2022")

<br><br>

---

<br>

## 4. Devolution: strategic authorities

England's **strategic authorities** (formerly combined authorities) take on powers from central government. The map file has each one's `tier` as a property, so we can colour by it directly. A grey layer of every council underneath fills in the areas without one.

In [ ]:
england_councils = alt.topo_feature(MAPS + "gbr/LAD_ENG_2026_05.json", "geog")
strategic_authorities = alt.topo_feature(MAPS + "gbr/SA_ENG_2026_07.json", "geog")

councils_layer = alt.Chart(england_councils).mark_geoshape(fill="#eeeeee", stroke="white", strokeWidth=0.3)
sa_layer = alt.Chart(strategic_authorities).mark_geoshape(stroke="white").encode(
    color=alt.Color("properties.tier:N").title("Tier").scale(
        domain=["established", "mayoral", "foundation"], range=["#0f5499", "#58b3c7", "#f4a582"]),
    tooltip=[alt.Tooltip("properties.name:N").title("Strategic authority"),
             alt.Tooltip("properties.tier:N").title("Tier")],
)

(councils_layer + sa_layer).project("mercator").properties(width=380, height=450, title="Devolution in England, July 2026")

<br>

### Zooming in on one strategic authority

map-data has a folder for each strategic authority, with its councils (`LAD.json`) and small areas (`MSOA.json`). Here: population density by MSOA in Greater Manchester, with council borders drawn on top as outlines only (`filled=False`).

To map another strategic authority, change the folder name. The list is in [map-data's guide](https://github.com/jhellingsdata/map-data/blob/main/gbr/sa/README.md).

In [ ]:
folder = MAPS + "gbr/sa/msa001_greater_manchester/"

small_areas = alt.Chart(alt.topo_feature(folder + "MSOA.json", "geog")).transform_lookup(
    lookup="properties.id",
    from_=alt.LookupData(DATA + "pop_density_census_msoa21.csv", key="geo_id", fields=["value"]),
).mark_geoshape(stroke="white", strokeWidth=0.2).encode(
    color=alt.Color("value:Q").title("People per km²").scale(type="log", scheme="oranges"),
    tooltip=[alt.Tooltip("properties.name:N").title("Area"),
             alt.Tooltip("value:Q").format(",.0f").title("People per km²")],
)

council_borders = alt.Chart(alt.topo_feature(folder + "LAD.json", "geog")).mark_geoshape(
    filled=False, stroke="#333333", strokeWidth=1
)

chart = (small_areas + council_borders).project("mercator").properties(
    width=400, height=380, title="Greater Manchester: population density"
)
chart

<br>

Save it as a JSON file, to embed on your website with `vegaEmbed` (as in Week 3):

In [ ]:
chart.save("greater_manchester_density.json")

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1.** Change the choropleth in section 3 to show another Welsh dataset from the `data` folder, such as `unemp_LAD_WLS.csv` (unemployment) or `gdp_per_head_LAD_WLS.csv`. These have many dates, so use the "many years" pattern. Hint: look at the file first to see how its dates are written.

**2.** Change the zoomed-in map in section 4 to a different strategic authority.

**3.** Colour Wales's councils by their Corporate Joint Committee region, using `wales_cjc_lookup.csv` (columns `lad_id`, `lad_name`, `cjc`). Hint: it's a lookup, but the value is text, so use `:N`.

**4. (Portfolio)** Make a map for **one** English strategic authority or for **Wales**, using a dataset you choose. The [`data/msa`](https://github.com/jhellingsdata/LSE-PP434/tree/main/weeks/week07/data/msa) folder has data for every strategic authority and council: its README says which map file to use. Add a clear title, a source, and a tooltip.

In [ ]:
# Your code here
